In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load the datasets
print("Loading datasets...")
df_sentiment = pd.read_csv('../data/fear_greed_index.csv')
df_trades = pd.read_csv('../data/historical_data.csv')

Loading datasets...


In [3]:
# 3. Standardize Dates

# Sentiment date is already YYYY-MM-DD. We just ensure it's a datetime object.
df_sentiment['date'] = pd.to_datetime(df_sentiment['date'])

# Trades data uses 'DD-MM-YYYY HH:MM' in IST. We extract just the date.
df_trades['Timestamp IST'] = pd.to_datetime(df_trades['Timestamp IST'], format='%d-%m-%Y %H:%M')
df_trades['date'] = df_trades['Timestamp IST'].dt.normalize()

In [5]:
# 4. Create Key Metrics (Aggregating to Daily Level per Account)
# We need: daily PnL, win rate, average trade size, number of trades, and long/short ratio

print("\nAggregating daily trader metrics...")

daily_trader_metrics = df_trades.groupby(['Account', 'date']).apply(
    lambda x: pd.Series({
        'daily_pnl': x['Closed PnL'].sum(),
        'total_trades': len(x),
        'winning_trades': (x['Closed PnL'] > 0).sum(),
        'avg_trade_size_usd': x['Size USD'].mean(),
        'long_trades': (x['Direction'] == 'Buy').sum(),
        'short_trades': (x['Direction'] == 'Sell').sum()
    })
).reset_index()


# Calculate derived metrics
daily_trader_metrics['win_rate'] = daily_trader_metrics['winning_trades'] / daily_trader_metrics['total_trades']

# Avoid division by zero for Long/Short ratio by adding a tiny number
daily_trader_metrics['long_short_ratio'] = daily_trader_metrics['long_trades'] / (daily_trader_metrics['short_trades'] + 1e-9)

print("done")


Aggregating daily trader metrics...
done


C:\Users\Yash Kumar\AppData\Local\Temp\ipykernel_12692\1397479837.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_trader_metrics = df_trades.groupby(['Account', 'date']).apply(


In [6]:
# 5. Merge Datasets
# Left join the sentiment data onto our daily trader metrics
final_df = pd.merge(daily_trader_metrics, df_sentiment[['date', 'value', 'classification']], on='date', how='left')

# Rename
final_df.rename(columns={'value': 'sentiment_score', 'classification': 'sentiment_label'}, inplace=True)

print("\n--- Merge Complete ---")
print(final_df.head())


--- Merge Complete ---
                                      Account       date  daily_pnl  \
0  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-11        0.0   
1  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-17        0.0   
2  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-18        0.0   
3  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-22   -21227.0   
4  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-26     1603.1   

   total_trades  winning_trades  avg_trade_size_usd  long_trades  \
0         177.0             0.0         5089.718249          0.0   
1          68.0             0.0         7976.664412          0.0   
2          40.0             0.0        23734.500000          0.0   
3          12.0             0.0        28186.666667          0.0   
4          27.0            12.0        17248.148148          0.0   

   short_trades  win_rate  long_short_ratio  sentiment_score sentiment_label  
0           0.0  0.000000               0.0             76.0 

In [8]:
# Save the prepared data 
final_df.to_csv('../data/prepared_trader_sentiment.csv', index=False)